In [ ]:
import torch
import torch.nn.functional as F


def gaussian_kernel1d(sigma, truncate=4.0, device=None, dtype=torch.float32):
    """1D Gaussian weights. Number of taps follows from sigma."""
    radius = int(truncate * sigma + 0.5)
    d = torch.arange(-radius, radius + 1, device=device, dtype=dtype)
    k = torch.exp(-d ** 2 / (2 * sigma ** 2))     # evaluate the formula
    return k / k.sum()                            # normalise: weights sum to 1


def _blur_one_axis(x, sigma, axis, truncate, pad_mode):
    """
    x: (N, 1, H, W) float tensor.
    axis=3 blurs along width (horizontal pass); axis=2 along height (vertical pass).
    """
    if sigma <= 0:                                # sigma 0 = no blur on this axis
        return x

    k = gaussian_kernel1d(sigma, truncate, x.device, x.dtype)
    r = k.numel() // 2

    if pad_mode == "reflect" and r >= x.shape[axis]:
        raise ValueError(
            f"kernel radius {r} must be smaller than the image size "
            f"{x.shape[axis]} along this axis; lower sigma/truncate or use "
            f"pad_mode='replicate'."
        )

    if axis == 3:                                 # horizontal pass
        x = F.pad(x, (r, r, 0, 0), mode=pad_mode)
        return F.conv2d(x, k.view(1, 1, 1, -1))   # kernel shape (1, 1, 1, K)
    else:                                         # vertical pass
        x = F.pad(x, (0, 0, r, r), mode=pad_mode)
        return F.conv2d(x, k.view(1, 1, -1, 1))   # kernel shape (1, 1, K, 1)


def gaussian_blur(x, sigma, truncate=4.0, pad_mode="reflect"):
    """
    x     : float tensor of shape (H, W), (C, H, W) or (B, C, H, W)
    sigma : float, or (sigma_y, sigma_x)
    Returns a tensor of the same shape as x.
    """
    orig_shape = x.shape
    while x.ndim < 4:                             # promote to (B, C, H, W)
        x = x.unsqueeze(0)
    B, C, H, W = x.shape

    sigma_y, sigma_x = (sigma, sigma) if isinstance(sigma, (int, float)) else sigma

    x = x.reshape(B * C, 1, H, W)                 # every channel becomes its own image
    x = _blur_one_axis(x, sigma_x, 3, truncate, pad_mode)   # pass 1: horizontal
    x = _blur_one_axis(x, sigma_y, 2, truncate, pad_mode)   # pass 2: vertical (chained)
    return x.reshape(orig_shape)


def blur_image(img_array, sigma, **kw):
    """
    Convenience wrapper for a uint8 numpy image, (H, W) or (H, W, C).
    Returns a uint8 numpy array of the same shape.
    """
    x = torch.from_numpy(img_array).float()
    if x.ndim == 3:
        x = x.permute(2, 0, 1)                    # (H, W, C) -> (C, H, W)

    out = gaussian_blur(x, sigma, **kw)

    if out.ndim == 3:
        out = out.permute(1, 2, 0)                # back to (H, W, C)
    return out.round().clamp(0, 255).byte().numpy()   # round, don't truncate

In [ ]:
def elastic_deform_coarse_torch(img, mask=None, points=3, sigma=10.0, generator=None):
    import torch
    import torch.nn.functional as F

    B, C, H, W = img.shape
    dev, dt = img.device, img.dtype

    # one coarse grid per sample; channel 0 = dy, channel 1 = dx
    coarse = torch.randn(B, 2, points, points, device=dev, dtype=dt, generator=generator) * sigma
    # cubic upsampling; align_corners=True puts anchors exactly on the corners
    disp = F.interpolate(coarse, size=(H, W), mode="bicubic", align_corners=True)   # (B, 2, H, W)
    dy, dx = disp[:, 0], disp[:, 1]

    ys, xs = torch.meshgrid(torch.arange(H, device=dev, dtype=dt),
                            torch.arange(W, device=dev, dtype=dt), indexing="ij")
    y, x = ys - dy, xs - dx                                       # where to read from
    grid = torch.stack([2 * x / (W - 1) - 1, 2 * y / (H - 1) - 1], dim=-1)   # (B,H,W,2), (x, y) in [-1,1]

    out_img = F.grid_sample(img, grid, mode="bilinear",
                            padding_mode="reflection", align_corners=True)
    if mask is None:
        return out_img
    squeeze = mask.ndim == 3
    m = (mask.unsqueeze(1) if squeeze else mask).to(dt)
    out_mask = F.grid_sample(m, grid, mode="nearest",             # SAME grid, nearest
                             padding_mode="reflection", align_corners=True).to(mask.dtype)
    return out_img, (out_mask.squeeze(1) if squeeze else out_mask)
